# Evaluating a knowledge edit — efficacy vs. specificity (Llama-3.2-1B, T4)

One prompt is easy to change. The hard part is changing it **without breaking the
neighbours**. Build a tagged battery, edit the `France -> Paris` constellation,
measure the trade-off, then sweep edit strength for the **Pareto frontier**.

| axis | editing-literature name | tag |
|---|---|---|
| did the target change? | **efficacy** | `target` |
| did related facts survive? | **specificity** | `neighbour` |
| did unrelated facts survive? | **locality** | `control` |

Runtime: **T4 GPU**. First cell ~ model download.


In [ ]:
!pip install -q 'transformers>=4.45' accelerate safetensors matplotlib
!git clone -q https://github.com/thebnbrkr/marv.git /content/marv
%cd /content/marv
!pip install -q -e .

## Load + extract


In [ ]:
import torch, numpy as np, marv
from transformers import AutoModelForCausalLM, AutoTokenizer
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Ungated Llama-3.2-1B mirror. Alternatives:
#   meta-llama/Llama-3.2-1B-Instruct   (needs an HF token)
#   Qwen/Qwen2.5-1.5B-Instruct         (bigger, stronger recall, slower)
NAME = 'unsloth/Llama-3.2-1B-Instruct'
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModelForCausalLM.from_pretrained(NAME, torch_dtype=torch.float16).to(device).eval()
print(model.config.num_hidden_layers, 'layers  hidden', model.config.hidden_size,
      ' intermediate', model.config.intermediate_size, ' vocab', model.config.vocab_size)

vindex = marv.extract(model, model_name=NAME)
marv.build_down_meta(vindex, device=device)   # GPU: ~seconds even at 128k vocab
print('bands:', vindex.layer_bands)

## The battery

`target` has rephrasings (an edit that only works on the exact prompt has poor
*generalization*). `neighbour` shares the constellation. `control` spans several
domains so damage anywhere shows up.


In [ ]:
P = marv.Probe
battery = [
    P('The capital of France is', 'Paris', ('target',)),
    P('The French capital is', 'Paris', ('target',)),
    P('Paris is the capital of', 'France', ('target',)),
    P('What is the capital of France? It is', 'Paris', ('target',)),

    P('The capital of Italy is', 'Rome', ('neighbour','capital')),
    P('The capital of Spain is', 'Madrid', ('neighbour','capital')),
    P('The capital of Germany is', 'Berlin', ('neighbour','capital')),
    P('The capital of Portugal is', 'Lisbon', ('neighbour','capital')),
    P('The official language of France is', 'French', ('neighbour','france')),
    P('The currency used in France is the', 'euro', ('neighbour','france')),
    P('The Eiffel Tower is in', 'Paris', ('neighbour','paris')),
    P('The Louvre is in', 'Paris', ('neighbour','paris')),

    P('The capital of Japan is', 'Tokyo', ('control','geo')),
    P('The capital of Egypt is', 'Cairo', ('control','geo')),
    P('The capital of Canada is', 'Ottawa', ('control','geo')),
    P('The largest planet in the solar system is', 'Jupiter', ('control','science')),
    P('Water is made of hydrogen and', 'oxygen', ('control','science')),
    P('The chemical symbol for gold is', 'Au', ('control','science')),
    P('The opposite of hot is', 'cold', ('control','lexical')),
    P('The past tense of go is', 'went', ('control','lexical')),
    P('Two plus two equals', 'four', ('control','math')),
    P('The author of Romeo and Juliet is', 'Shakespeare', ('control','culture')),
    P('The first man on the moon was', 'Neil', ('control','history')),
    P('A group of wolves is called a', 'pack', ('control','lexical')),
]
print(len(battery), 'probes')

## Baseline

Keep only facts the model gets right (rank 1) unedited -- an edit eval is
meaningless on facts it doesn't know.


In [ ]:
base = marv.run_battery(model, tok, battery, device=device)
for r in base.rows:
    m = 'ok' if r.target_rank == 1 else f'r{r.target_rank}'
    print(f'  [{m:>4}] p={r.target_prob:.2f}  {r.prompt!r} -> {r.top1!r}  {r.tags}')

known = {r.prompt for r in base.rows if r.target_rank <= 3}
battery = [p for p in battery if p.prompt in known]
print(f'\nkept {len(battery)} probes with target rank <= 3')

## Locate the constellation — causally

1. contextual gate-KNN gives a **candidate pool**.
2. `rank_by_ablation_effect` suppresses each candidate alone and ranks by the
   measured drop in target probability -- the causal constellation.


In [ ]:
pool = marv.constellation(vindex, tok, 'France', model=model,
                          prompt='The capital of France is',
                          baseline_prompt='The capital of',
                          per_layer=8, device=device)
target_probes = [p for p in battery if 'target' in p.tags]
ranked = marv.rank_by_ablation_effect(
    model, tok, [(r.layer, r.feature) for r in pool[:30]], target_probes, device=device)
for (L, f), drop in ranked[:15]:
    tks,_ = marv.describe_feature(vindex, L, f, k=3)
    print(f'  L{L:>2} f{f:<5} drop={drop:+.3f}  -> {[w.strip() for w in tok.batch_decode([[int(t)] for t in tks])]}')
feats = [c for c, _ in ranked]

## One edit: suppress the top 5


In [ ]:
rep = marv.study_edit(model, tok, marv.suppress(model, feats[:5]), battery, device=device)
rep.show()
print()
for tag, m in sorted(rep.metrics().items()):
    print(f'  {tag:<12} n={int(m["n"]):>2}  moved={m["moved"]:.2f}  mean dprob={m["mean_dprob"]:+.3f}')

## The frontier


In [ ]:
sizes = [0, 1, 2, 3, 4, 6, 8, 10, 14, 20]
sweep = marv.suppression_frontier(model, tok, feats, battery, sizes=sizes, device=device)
rows = []
for n, d in sweep:
    m = d.metrics()
    g = lambda t, k: m.get(t, {}).get(k, 0.0)
    rows.append((n, g('target','mean_dprob'), g('neighbour','mean_dprob'), g('control','mean_dprob'),
                 g('neighbour','moved'), g('control','moved')))
print(f'{"n":>3} {"target dp":>10} {"neigh dp":>10} {"ctrl dp":>10} {"neigh mv":>9} {"ctrl mv":>8}')
for r in rows:
    print(f'{r[0]:>3} {r[1]:>+10.3f} {r[2]:>+10.3f} {r[3]:>+10.3f} {r[4]:>9.2f} {r[5]:>8.2f}')

In [ ]:
import matplotlib.pyplot as plt
ns = [r[0] for r in rows]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ns, [-r[1] for r in rows], 'o-', label='target (efficacy)')
ax[0].plot(ns, [-r[2] for r in rows], 's-', label='neighbour (collateral)')
ax[0].plot(ns, [-r[3] for r in rows], '^-', label='control (collateral)')
ax[0].set_xlabel('features suppressed'); ax[0].set_ylabel('mean target-prob drop')
ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot([-r[2] for r in rows], [-r[1] for r in rows], 'o-')
for r in rows: ax[1].annotate(str(r[0]), (-r[2], -r[1]), fontsize=8, xytext=(3,3), textcoords='offset points')
ax[1].set_xlabel('neighbour prob drop (collateral)'); ax[1].set_ylabel('target prob drop (efficacy)')
ax[1].set_title('Pareto frontier'); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## suppress vs ablate vs steer


In [ ]:
ef = feats[:5]
sup = marv.study_edit(model, tok, marv.suppress(model, ef), battery, device=device).metrics()
b0 = marv.run_battery(model, tok, battery, device=device)
saved = marv.ablate(model, ef)
abl = marv.diff_battery(b0, marv.run_battery(model, tok, battery, device=device)).metrics()
marv.restore(model, saved)
pid = tok.encode(' Paris', add_special_tokens=False)[0]
kb = vindex.band('knowledge'); mid = kb[len(kb)//2]
ste = marv.study_edit(model, tok, marv.steer(model, mid, vindex.embed[pid].astype(np.float32), alpha=-10.0), battery, device=device).metrics()
print(f'{"":<10} {"target dp":>10} {"neigh dp":>10} {"ctrl dp":>10}')
for name, m in [('suppress', sup), ('ablate', abl), ('steer', ste)]:
    g = lambda t: m.get(t, {}).get('mean_dprob', 0.0)
    print(f'{name:<10} {g("target"):>+10.3f} {g("neighbour"):>+10.3f} {g("control"):>+10.3f}')

## Reading it

- **Knee** in the frontier (target drops, neighbours flat) = a clean edit exists.
  **Straight line through origin** = fact and neighbours share features, no clean edit.
- `ablate` should track `suppress`. `steer` is usually blunter.
- Paper-grade eval adds: multi-hop consequences (MQuAKE), a bigger control set,
  several target facts averaged, and a fluency check on free generation.

### Try
- Swap `France` for `Liechtenstein` or a fictional place -- collateral usually
  collapses because the constellation is barely shared.
- Rerun with `Qwen/Qwen2.5-1.5B-Instruct` and compare frontiers across model families.
